## Group-Level Analysis

So far, we've analyzed data from a single subject. To make inferences about the population, we need to perform group-level analyses. This typically involves:

1. **First-level analysis**: Fit GLM for each subject (what we just did)
2. **Second-level analysis**: Combine results across subjects

Let's demonstrate this with multiple subjects:

In [ ]:
import numpy as np
from nltools.datasets import fetch_haxby

## Loading the Haxby Dataset

Let's start by loading the Haxby dataset. The `fetch_haxby()` function downloads and loads the data for us, returning both the fMRI data and design matrices.

In [ ]:
# Load data for multiple subjects
all_subjects_data, all_subjects_dm = fetch_haxby(n_subjects="all", verbose=1)

print(f"Loaded {len(all_subjects_data)} subjects")
print(f"Subject 1 has {len(all_subjects_data[0])} runs")

Loaded 6 subjects
Subject 1 has 12 runs


For group analysis, we would:
1. Fit GLM for each subject (first-level)
2. Extract contrast maps for each subject
3. Perform a one-sample t-test across subjects (second-level)

Here's a simplified example:

In [ ]:
# Example: Extract face > house contrast for first subject, first run
# In practice, you'd do this for all subjects and runs
subject1_run1 = all_subjects_data[0][0]
subject1_dm = all_subjects_dm[0][0]

# Add nuisance variables
subject1_dm_filt = subject1_dm.add_dct_basis(duration=128).add_poly(
    order=2, include_lower=True
)

# Fit GLM
subject1_run1.fit(model="glm", X=subject1_dm_filt)

# Compute contrast
face_vs_house_sub1 = subject1_run1.compute_contrasts("face - house")

print("Contrast computed for subject 1, run 1")
face_vs_house_sub1.plot(title="Subject 1: Face > House")

## Thresholding and Multiple Comparisons

When examining contrast maps, we need to account for multiple comparisons (testing thousands of voxels). Common approaches include:

1. **Uncorrected thresholding**: Simple p-value threshold (e.g., p < 0.001)
2. **False Discovery Rate (FDR)**: Controls the expected proportion of false positives
3. **Family-Wise Error Rate (FWER)**: Controls the probability of any false positives

The `BrainData` class provides methods for thresholding:

In [ ]:
# Example: Threshold the contrast map
# Note: This is a simplified example. In practice, you'd use proper statistical thresholding

# Get t-statistic map (compute_contrasts returns t-statistics by default)
t_map = face_vs_house

# Simple thresholding (uncorrected)
# In practice, you'd use proper FDR or cluster correction
thresholded = t_map.copy()
thresholded.data[np.abs(thresholded.data) < 2.0] = 0  # Threshold at |t| > 2

thresholded.plot(title="Thresholded Contrast Map (|t| > 2)")